<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Breakout/Resistance_Breakout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Resistance Breakout Strategy & Performance Scanner

This notebook implements a technical analysis scanner designed to identify **Resistance Breakouts**. The strategy focuses on identifying stocks that establish a clear price ceiling and subsequently break through it with high momentum.

**Key Features:**
* **Dynamic Resistance Detection:** Identifies price levels where a ticker has 'touched' or peaked multiple times (configurable via `min_touches`) within a lookback window.
* **Breakout Validation:** Triggers a signal only when the price closes above the resistance level by a specified percentage buffer and sets a new local high.
* **Ticker Locking:** Prevents redundant signals by enforcing a 'cool-off' period (e.g., 10 days) after a breakout is detected for a specific ticker.
* **Forward Performance Tracking:** Automatically calculates the 1-day through 5-day returns following each breakout and tracks the 'Success Rate' (percentage of time the price remains above the breakout level).
* **Buffered Data Ingestion:** Automatically downloads an extra 10 days of historical data beyond the scan window to ensure end-of-year signals have complete performance data.

**Current Configuration:** Scanning high-volume tickers for breakouts between January 2025 and December 2025.

In [79]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from datetime import timedelta

In [80]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: csv_input
Deleted DataFrame: full_historical_data
Deleted DataFrame: watchlist_df
Deleted DataFrame: df
Deleted DataFrame: recent_df
Deleted DataFrame: performance_df
Deleted DataFrame: valid_subset
All DataFrames cleared from memory.


In [81]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### 1. Configuration and Data Ingestion
This cell defines the date range for the scan and calculates a 10-day 'buffer' for `DATA_END_DATE` to ensure we can track performance for year-end breakouts. It then loads the ticker list from Google Drive and downloads the necessary historical data via `yfinance`.

In [82]:
# 1. Parameterize scan range and download buffer
START_SCAN = '2025-01-01'
END_SCAN = '2025-12-31'  # The period evaluated for breakouts

# Calculate a data download end date exactly 10 days past the scan period
# to ensure full forward-performance data for year-end signals.
END_SCAN_DT = pd.to_datetime(END_SCAN)
DATA_END_DATE = (END_SCAN_DT + timedelta(days=10)).strftime('%Y-%m-%d')

print(f"Scan range: {START_SCAN} to {END_SCAN}")
print(f"Downloading data through: {DATA_END_DATE} (10-day performance buffer)")

# 2. Ingest list of tickers from your CSV file
tickers = []
try:
    csv_input = pd.read_csv(OptionVolume)
    tickers = csv_input['Symbol'].dropna().unique().tolist()
    tickers = [str(t).strip().upper() for t in tickers if str(t).isalpha()]
except Exception as e:
    print(f"Error reading CSV: {e}. Falling back to test list.")
    tickers = ["AAPL", "MSFT", "AMD", "NVDA", "INTC", "TSLA"]

# 3. Batch download historical data once using the buffered date
full_historical_data = pd.DataFrame()
if tickers:
    print(f"Downloading historical data for {len(tickers)} tickers...")
    try:
        full_historical_data = yf.download(tickers, start=START_SCAN, end=DATA_END_DATE, group_by='ticker', auto_adjust=True)
        print("Historical data download complete.")
    except Exception as e:
        print(f"Error fetching data: {e}")

if full_historical_data.empty:
    print("No historical data available for analysis.")

Scan range: 2025-01-01 to 2025-12-31


[*********************100%***********************]  99 of 99 completed

Historical data download complete.


### 2. Resistance Breakout Scanner
This function iterates through the historical data to find price levels where a stock has peaked multiple times. It triggers a signal when the price decisively closes above that resistance level. It also includes a 'cool-off' mechanism to prevent redundant signals for the same stock.

In [83]:
def scan_resistance_breakout(ticker_list, full_data,
                             start_date_str, end_date_str,
                             resistance_buffer_pct=0.005, min_touches=3,
                             lookback_window=60, breakout_buffer_pct=0.01,
                             cool_off_days=30):
    """
    Scans for unique resistance breakouts between start_date_str and end_date_str.
    """
    candidates = []
    last_breakout_date = {}

    # Convert strings to datetime objects for comparison
    start_dt = pd.to_datetime(start_date_str)
    end_dt = pd.to_datetime(end_date_str)

    print(f"Scanning {len(ticker_list)} tickers for triggers between {start_date_str} and {end_date_str}...")

    for ticker in ticker_list:
        try:
            if ticker not in full_data.columns.levels[0]:
                continue
            df = full_data[ticker].dropna().copy()
            if df.empty or len(df) < lookback_window + 2:
                continue

            last_breakout_date[ticker] = None

            for i in range(lookback_window, len(df)):
                current_date = df.index[i]

                # Filter: Only identify triggers within the defined scan window
                if current_date < start_dt or current_date > end_dt:
                    continue

                # Check if we are within the cool-off period
                if last_breakout_date[ticker] is not None:
                    days_since = (current_date - last_breakout_date[ticker]).days
                    if days_since < cool_off_days:
                        continue

                recent_df = df.iloc[i-lookback_window : i]
                current_close = df['Close'].iloc[i]
                previous_close = df['Close'].iloc[i-1]
                highest_high_in_window = recent_df['High'].max()

                # 1. Identify Resistance Levels
                highs = recent_df['High'].values
                sorted_highs = np.sort(highs)
                identified_levels = []

                for val in sorted_highs:
                    cluster = [h for h in highs if abs(h - val) / val <= resistance_buffer_pct]
                    if len(cluster) >= min_touches:
                        res_level = np.mean(cluster)
                        if not any(abs(res_level - lvl) / lvl < 0.02 for lvl, _ in identified_levels):
                            identified_levels.append((res_level, len(cluster)))

                # 2. Check Breakout Conditions
                for res_price, touches in identified_levels:
                    breakout_threshold = res_price * (1 + breakout_buffer_pct)

                    if (current_close > breakout_threshold) and (previous_close <= breakout_threshold):
                        if current_close > highest_high_in_window:
                            candidates.append({
                                "Ticker": ticker,
                                "Date": current_date.strftime('%Y-%m-%d'),
                                "Res_Level": round(res_price, 2),
                                "Touches": touches,
                                "Close": round(current_close, 2),
                                "Prev_High": round(highest_high_in_window, 2)
                            })
                            last_breakout_date[ticker] = current_date
                            break

        except Exception:
            continue

    return pd.DataFrame(candidates)

# Re-run the scan using the parameter window
if 'full_historical_data' in globals() and not full_historical_data.empty:
    watchlist_df = scan_resistance_breakout(
        tickers,
        full_historical_data,
        start_date_str=START_SCAN,
        end_date_str=END_SCAN,
        min_touches=5,
        cool_off_days=10
    )
    print(f"\nTotal triggers identified: {len(watchlist_df)}")
    display(watchlist_df.head(20))

Scanning 99 tickers for triggers between 2025-01-01 and 2025-12-31...

Total triggers identified: 191


,Ticker,Date,Res_Level,Touches,Close,Prev_High
0,SPY,2025-06-03,583.00,6,589.33,588.79
1,SPY,2025-06-24,594.85,10,601.68,598.20
2,SPY,2025-07-17,614.20,5,622.76,622.58
3,SPY,2025-08-12,627.87,6,637.28,634.47
4,SPY,2025-09-11,640.56,12,652.10,649.04
5,SPY,2025-09-22,654.84,5,663.06,660.79
6,TSLA,2025-09-11,356.28,5,368.81,358.44
7,QQQ,2025-06-03,517.40,7,524.76,523.94
8,QQQ,2025-06-24,528.36,5,537.78,534.19
9,QQQ,2025-07-17,550.53,5,559.72,558.73


### 3. Performance Analysis and Validation
This final section calculates the forward returns for 1 to 5 days following every identified breakout. It generates a summary table showing the average returns and a 'Success Rate,' verifying how often the price stayed above the breakout level.

In [84]:
def calculate_post_breakout_performance(watchlist, full_data):
    results = []

    for _, row in watchlist.iterrows():
        ticker = row['Ticker']
        breakout_date = row['Date']
        res_level = row['Res_Level']
        breakout_close = row['Close']

        if ticker not in full_data.columns.levels[0]:
            continue

        df = full_data[ticker].dropna()
        try:
            idx = df.index.get_loc(breakout_date)
        except KeyError:
            continue

        performance = row.to_dict()

        # Loop specifically for Days 1 through 5
        for d in [1, 2, 3, 4, 5]:
            future_idx = idx + d
            if future_idx < len(df):
                future_close = df['Close'].iloc[future_idx]
                ret = ((future_close - breakout_close) / breakout_close) * 100
                performance[f'Day_{d}_Return_%'] = round(ret, 2)
                performance[f'Day_{d}_Above_Res'] = bool(future_close >= res_level)
            else:
                performance[f'Day_{d}_Return_%'] = np.nan
                performance[f'Day_{d}_Above_Res'] = None

        results.append(performance)

    return pd.DataFrame(results)

if 'watchlist_df' in globals() and not watchlist_df.empty:
    performance_df = calculate_post_breakout_performance(watchlist_df, full_historical_data)

    # Calculate Summary for Days 1-5
    summary_data = {}
    validation_log = []

    for d in [1, 2, 3, 4, 5]:
        col_name = f'Day_{d}_Above_Res'
        valid_subset = performance_df[performance_df[col_name].notnull()]

        total_valid = len(valid_subset)
        success_count = valid_subset[col_name].sum() if total_valid > 0 else 0
        fail_count = total_valid - success_count

        avg_ret = valid_subset[f'Day_{d}_Return_%'].mean() if total_valid > 0 else np.nan
        pct_above = (success_count / total_valid * 100) if total_valid > 0 else 0

        summary_data[f'Day_{d} Avg Return %'] = avg_ret
        summary_data[f'Day_{d} % Above Res'] = pct_above

        validation_log.append({"Day": d, "Total": total_valid, "Above": success_count, "Below": fail_count})

    print("--- Updated Performance Summary (with 10-day buffer) ---")
    display(pd.Series(summary_data).to_frame(name='Value'))

    print("\n--- Raw Verification Counts ---")
    display(pd.DataFrame(validation_log))

    print("\n--- Detailed Performance Table (Full) ---")
    # Using pd.option_context to ensure the full table is visible if desired,
    # or simply display() for the standard scrollable interactive table.
    display(performance_df)

--- Updated Performance Summary (with 10-day buffer) ---


,Value
Day_1 Avg Return %,0.383194
Day_1 % Above Res,93.193717
Day_2 Avg Return %,0.614503
Day_2 % Above Res,89.528796
Day_3 Avg Return %,1.006021
Day_3 % Above Res,87.434555
Day_4 Avg Return %,0.872094
Day_4 % Above Res,84.293194
Day_5 Avg Return %,1.166178
Day_5 % Above Res,81.675393



--- Raw Verification Counts ---


,Day,Total,Above,Below
0,1,191,178,13
1,2,191,171,20
2,3,191,167,24
3,4,191,161,30
4,5,191,156,35



--- Detailed Performance Table (Full) ---


,Ticker,Date,Res_Level,Touches,Close,Prev_High,Day_1_Return_%,Day_1_Above_Res,Day_2_Return_%,Day_2_Above_Res,Day_3_Return_%,Day_3_Above_Res,Day_4_Return_%,Day_4_Above_Res,Day_5_Return_%,Day_5_Above_Res
0,SPY,2025-06-03,583.00,6,589.33,588.79,-0.03,True,-0.51,True,0.51,True,0.60,True,1.17,True
1,SPY,2025-06-24,594.85,10,601.68,598.20,0.06,True,0.84,True,1.34,True,1.82,True,1.79,True
2,SPY,2025-07-17,614.20,5,622.76,622.58,-0.07,True,0.12,True,0.13,True,0.98,True,1.02,True
3,SPY,2025-08-12,627.87,6,637.28,634.47,0.34,True,0.35,True,0.12,True,0.10,True,-0.45,True
4,SPY,2025-09-11,640.56,12,652.10,649.04,-0.03,True,0.50,True,0.36,True,0.24,True,0.70,True
5,SPY,2025-09-22,654.84,5,663.06,660.79,-0.54,True,-0.86,True,-1.32,False,-0.75,True,-0.47,True
6,TSLA,2025-09-11,356.28,5,368.81,358.44,7.36,True,11.18,True,14.32,True,15.47,True,13.03,True
7,QQQ,2025-06-03,517.40,7,524.76,523.94,0.28,True,-0.48,True,0.50,True,0.64,True,1.31,True
8,QQQ,2025-06-24,528.36,5,537.78,534.19,0.26,True,1.19,True,1.54,True,2.20,True,1.34,True
9,QQQ,2025-07-17,550.53,5,559.72,558.73,-0.10,True,0.42,True,-0.10,True,0.36,True,0.57,True
